In [ ]:
# --- SETUP (same in every notebook) -----------------------------------
OPERATOR = "nikolai"                       # <- your name (your own working copy of the code)

from google.colab import drive
drive.mount('/content/drive')
import os, sys, subprocess
ROOT = "/content/drive/MyDrive/pei_landuse"
REPO = f"{ROOT}/code/{OPERATOR}/pei-landuse"
assert os.path.exists(REPO), "No code yet: run notebooks/00_git.ipynb (clone) first."
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", f"{REPO}/requirements-colab.txt"], check=True)
sys.path.insert(0, f"{REPO}/src"); os.chdir(REPO)
%load_ext autoreload
%autoreload 2
print(subprocess.run("git branch --show-current; git log -1 --oneline; git status --short | head", shell=True, capture_output=True, text=True).stdout)

In [ ]:
CONFIG = "configs/remerge_1968_prince_hedgerow.yaml"
X, Y = 0.0, 0.0          # <- map coordinates of a problem spot (from ArcGIS)
TRY = {}                 # e.g. {"rag_threshold": 0.25, "w_dark": 1.5, "line_radius_px": 8}
RUN_ID = ""              # run folder with raw detections (ignored if the config sets paths.raw_detections)

import matplotlib.pyplot as plt, numpy as np
from pei.config import load_config
from pei.infer import load_raw_detections, resolve_raster
from pei.segment import debug_area

cfg = load_config(CONFIG)
run_dir = f"{cfg['paths']['runs']}/_tuning"; os.makedirs(run_dir, exist_ok=True)
raster = resolve_raster(cfg, run_dir)
raw = load_raw_detections(f"{cfg['paths']['runs']}/{RUN_ID}", cfg["paths"].get("raw_detections"))

In [ ]:
polys, d = debug_area(cfg, raw, raster, X, Y, overrides=TRY)
shuffle = lambda L: np.random.default_rng(0).permutation(L.max() + 1)[L] * (L > 0)
fig, ax = plt.subplots(2, 3, figsize=(20, 13))
for a, (im, t, cm) in zip(ax.ravel(), [
        (d["img"], "image", "gray"), (d["strength"], "line strength (hedgerows bright)", "magma"),
        (d["markers"], "seeds (one per detection, duplicates dropped)", "tab20"),
        (shuffle(d["watershed"]), "watershed", "tab20"), (shuffle(d["labels"]), "after re-joining (final)", "tab20"),
        (d["basin"], "clear-land basin", "gray")]):
    a.imshow(im, cmap=cm, interpolation="nearest"); a.set_title(t); a.axis("off")
plt.tight_layout(); plt.show()
print(len(polys), "fields in this block")